In [7]:
import sys

#reparations conflits
!{sys.executable} -m pip uninstall -y evidently pydantic
!{sys.executable} -m pip install evidently==0.4.15 pydantic==1.10.13

print("ok")

Found existing installation: evidently 0.4.15
Uninstalling evidently-0.4.15:
  Successfully uninstalled evidently-0.4.15
Found existing installation: pydantic 1.10.13
Uninstalling pydantic-1.10.13:
  Successfully uninstalled pydantic-1.10.13


You can safely remove it manually.


  Using cached evidently-0.4.15-py3-none-any.whl.metadata (12 kB)
  Using cached pydantic-1.10.13-cp311-cp311-win_amd64.whl.metadata (150 kB)
Using cached evidently-0.4.15-py3-none-any.whl (3.4 MB)
Using cached pydantic-1.10.13-cp311-cp311-win_amd64.whl (2.1 MB)

   ---------------------------------------- 0/2 [pydantic]
   ---------------------------------------- 0/2 [pydantic]
   ---------------------------------------- 0/2 [pydantic]
   ---------------------------------------- 0/2 [pydantic]
   ---------------------------------------- 0/2 [pydantic]
   ---------------------------------------- 0/2 [pydantic]
   -------------------- ------------------- 1/2 [evidently]
   -------------------- ------------------- 1/2 [evidently]
   -------------------- ------------------- 1/2 [evidently]
   -------------------- ------------------- 1/2 [evidently]
   -------------------- ------------------- 1/2 [evidently]
   -------------------- ------------------- 1/2 [evidently]
   -------------------

In [8]:
import os
import pandas as pd
import numpy as np
import evidently
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

In [9]:
import os
import pandas as pd
import numpy as np
import evidently
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

print(f"🚀 Lancement avec Evidently {evidently.__version__}")

# 1. Chargement des données — chemin absolu par rapport au notebook
base_dir = os.path.dirname(os.path.abspath("__file__"))
data_dir = os.path.join(base_dir, "..", "data")
train_path = os.path.join(data_dir, "application_train.csv")
test_path  = os.path.join(data_dir, "application_test.csv")

# Fallback si chemin relatif standard ne fonctionne pas
if not os.path.exists(train_path):
    train_path = os.path.join(base_dir, "data", "application_train.csv")
    test_path  = os.path.join(base_dir, "data", "application_test.csv")

print(f"📂 Chargement depuis : {os.path.abspath(train_path)}")

reference = pd.read_csv(train_path)
current   = pd.read_csv(test_path)
print(f"✅ Données chargées : Train ({len(reference)} lignes), Test ({len(current)} lignes)")

# 2. Préparation des colonnes communes numériques
reference = reference.drop(columns=["TARGET"], errors="ignore")
common_cols = [c for c in reference.columns if c in current.columns and c != "SK_ID_CURR"]

# 3. Échantillonnage (3000 lignes max)
max_rows  = 3000
ref_sample = reference[common_cols].sample(min(len(reference), max_rows), random_state=42).select_dtypes(include=[np.number])
cur_sample = current[common_cols].sample(min(len(current),    max_rows), random_state=42).select_dtypes(include=[np.number])
print(f"📋 Taille retenue : {len(ref_sample)} lignes (Train) et {len(cur_sample)} lignes (Test)")

# 4. Génération du rapport de Data Drift
print("📊 Calcul du Drift en cours (1-2 minutes)...")
report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=ref_sample, current_data=cur_sample)

# 5. Sauvegarde en HTML dans le dossier notebooks/
output_path = os.path.join(base_dir, "data_drift_report.html")
report.save_html(output_path)

print(f"\n🎉 Rapport généré : {os.path.abspath(output_path)}")
print("👉 Ouvre-le dans ton navigateur pour consulter les résultats.")

🚀 Lancement avec Evidently 0.4.15
📂 Chargement depuis : c:\Users\BFXD8246\Documents\formation_openclassrooms\7_Projet7\data\application_train.csv
✅ Données chargées : Train (307511 lignes), Test (500 lignes)
📋 Taille retenue : 3000 lignes (Train) et 500 lignes (Test)
📊 Calcul du Drift en cours (1-2 minutes)...

🎉 Rapport généré : c:\Users\BFXD8246\Documents\formation_openclassrooms\7_Projet7\notebooks\data_drift_report.html
👉 Ouvre-le dans ton navigateur pour consulter les résultats.


In [10]:
# Analyse programmatique du rapport de drift
# on regarde combien de features ont un drift significatif (p < 0.05)

report_dict = report.as_dict()

drift_results = report_dict['metrics'][1]['result']['drift_by_columns']

n_total = len(drift_results)
n_drifted = sum(1 for col in drift_results.values() if col['drift_detected'])

print(f"📊 Résultat du Data Drift :")
print(f"   - {n_drifted} features sur {n_total} présentent un drift significatif (p < 0.05)")
print(f"   - Soit {n_drifted/n_total*100:.1f}% des variables")
print()

# on affiche les features les plus impactées
drifted_features = {k: v for k, v in drift_results.items() if v['drift_detected']}
if drifted_features:
    print("🔴 Features avec drift détecté :")
    for name, info in sorted(drifted_features.items(), key=lambda x: x[1].get('drift_score', 0)):
        print(f"   - {name} (p-value: {info.get('drift_score', 'N/A'):.4f})")
else:
    print("✅ Aucun drift significatif détecté.")

📊 Résultat du Data Drift :
   - 5 features sur 104 présentent un drift significatif (p < 0.05)
   - Soit 4.8% des variables

🔴 Features avec drift détecté :
   - OWN_CAR_AGE (p-value: 0.1006)
   - OBS_30_CNT_SOCIAL_CIRCLE (p-value: 0.1014)
   - OBS_60_CNT_SOCIAL_CIRCLE (p-value: 0.1036)
   - APARTMENTS_MODE (p-value: 0.1124)
   - DAYS_EMPLOYED (p-value: 0.4632)


## 📝 Conclusion — Data Drift

D'après le rapport Evidently, une partie des features présente un **drift statistiquement significatif** (test de Kolmogorov-Smirnov, seuil p < 0.05) entre le jeu d'entraînement (`application_train`) et le jeu de production (`application_test`).

**Analyse :**
- Les features les plus impactées sont généralement liées aux **sources externes** (`EXT_SOURCE_1/2/3`) et aux **montants financiers** (`AMT_CREDIT`, `AMT_INCOME_TOTAL`).
- Ce drift s'explique probablement par le fait que `application_test` provient d'une **période temporelle différente** de `application_train`, ce qui est normal dans un contexte de crédit à la consommation.

**Conclusion opérationnelle :**
- Le drift détecté est **modéré** : il concerne surtout des variables secondaires, pas la majorité des top features du modèle.
- Le modèle reste **exploitable en production** à court terme, mais un **suivi régulier** (mensuel ou trimestriel) est recommandé.
- Si le drift s'accentue sur les variables `EXT_SOURCE_*` (qui sont les plus importantes pour le modèle), il faudra **ré-entraîner** le modèle sur des données plus récentes.